# Modelo predictor de tipo de fraude

El modelo hereda el mismo pipeline de preprocesamiento aplicado a los modelos originales.

**Input esperado**: un DataFrame `df_etiquetado` con las columnas del esquema Ánfora v6
más una columna `cluster_fraude` (0 o 1 para fraudes, NaN para legítimas).

**Output**: dict `modelos_retrained`

## 0. Dependencias

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier
import warnings
import joblib
import os

warnings.filterwarnings('ignore')

SEED = 20260424
np.random.seed(SEED)

## 1. Descarga de modelo original

In [2]:
model = joblib.load("datos/v6/modelos/M3_gbm_completo.pkl")

In [3]:
model

{'pipeline': Pipeline(steps=[('pre',
                  ColumnTransformer(transformers=[('num', 'passthrough',
                                                   ['monto_log', 'secuencia_24h',
                                                    'terminal_risk_score']),
                                                  ('cat',
                                                   OneHotEncoder(handle_unknown='ignore'),
                                                   ['canal', 'dispositivo',
                                                    'hora_bin'])])),
                 ('clf',
                  GradientBoostingClassifier(learning_rate=0.08, max_depth=4,
                                             n_estimators=200,
                                             random_state=20260424,
                                             subsample=0.8))]),
 'features': ['monto_log',
  'secuencia_24h',
  'terminal_risk_score',
  'canal',
  'dispositivo',
  'hora_bin'],
 'threshold': 0.46}

## 2. Carga de datos etiquetados con cluster

- `es_fraude` (bool)
- `cluster_fraude` (0 para fraudes tipicos en T0 y 1 para nuevos fraudes en T1)

In [4]:
# Primero el dataset de operaciones
df = pd.read_parquet("datos/v6/operaciones.parquet")
df = df[df['etiquetado'] == True]

# Despues el que incluye los Cluster con los tipos de fraude
df_cluster = pd.read_parquet("datos/v6/clustering/clustering_operaciones.parquet")
df_cluster = df_cluster[df_cluster['cluster_fraude'].notna()]

# Hacemos un merge para tener toda la información en un mismo dataframe
df_merger = df.merge(df_cluster[["op_id", "cluster_fraude"]], on="op_id", how="inner")

In [5]:
df_merger['etiquetado'].value_counts()

etiquetado
True    5786
Name: count, dtype: int64

In [6]:
# Chequeamos la distribución de fraudes y tipos de fraude en el dataset de entrenamiento

assert 'es_fraude' in df_merger.columns
assert 'cluster_fraude' in df_merger.columns

n_fraude = df_merger['es_fraude'].sum()
n_legitimas = (~df_merger['es_fraude']).sum()
n_tipo0 = (df_merger['cluster_fraude'] == 0).sum()
n_tipo1 = (df_merger['cluster_fraude'] == 1).sum()

print(f"Total registros : {len(df_merger):,}")
print(f"Fraudes totales : {n_fraude:,}  ({n_fraude/len(df_merger)*100:.2f}%)")
print(f"  cluster 0     : {n_tipo0:,}")
print(f"  cluster 1     : {n_tipo1:,}")
print(f"Legítimas       : {n_legitimas:,}")

Total registros : 5,786
Fraudes totales : 5,786  (100.00%)
  cluster 0     : 4,781
  cluster 1     : 1,005
Legítimas       : 0


In [7]:
# 1. Extraer el pipeline del diccionario
pipeline = model['pipeline']

# 2. Extraer el ColumnTransformer usando su nombre de paso ('pre')
preprocesador = pipeline.named_steps['pre']
columnas_esperadas = preprocesador.feature_names_in_

# 1. Definir tus variables X (cruda) e y
X_crudo = df_merger[columnas_esperadas]
y = df_merger['cluster_fraude']

# 2. Hacer el split (en este caso un 10% para testeo)
# stratify=y es clave para mantener la proporción de fraude en ambos sets
X_train, X_test, y_train, y_test = train_test_split(
    X_crudo,
    y,
    test_size=0.05,          # 0.10 para el 10% o 0.05 para el 5%
    random_state=20260424,   # Mantenemos tu misma semilla
    stratify=y               # Mantiene la proporción de clases equilibrada
)

print(f"Tamaño Entrenamiento: {X_train.shape[0]} filas")
print(f"Tamaño Testeo:        {X_test.shape[0]} filas")
print(f"Proporción de fraude en Testeo: {y_test.mean():.4f}")


Tamaño Entrenamiento: 5496 filas
Tamaño Testeo:        290 filas
Proporción de fraude en Testeo: 0.1724


In [8]:
# 2. Definir el clasificador base
model_smishing = GradientBoostingClassifier(
    random_state=20260424,
    subsample=0.8
)

# 3. Encapsular preprocesador y modelo en un único Pipeline
pipeline_completo = Pipeline(steps=[
    ('pre', preprocesador),
    ('clf', model_smishing)
])

# 4. Definir la grilla de hiperparámetros
# (Se agrega el prefijo 'clf__' para apuntar al clasificador dentro del pipeline)
param_grid = {
    'clf__n_estimators': [100, 200, 300],
    'clf__learning_rate': [0.05, 0.08, 0.1],
    'clf__max_depth': [3, 4, 5]
}

# 5. Configurar el Grid Search apuntando al pipeline completo
grid_search_sm = GridSearchCV(
    estimator=pipeline_completo,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

# 7. Ejecutar la búsqueda con los datos crudos
grid_search_sm.fit(X_train, y_train)

Fitting 5 folds for each of 27 candidates, totalling 135 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...sample=0.8))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'clf__learning_rate': [0.05, 0.08, ...], 'clf__max_depth': [3, 4, ...], 'clf__n_estimators': [100, 200, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is

In [9]:
# 2. Definir el clasificador base
model_base = GradientBoostingClassifier(
    random_state=20260424,
    subsample=0.8
)

# 3. Encapsular preprocesador y modelo en un único Pipeline
pipeline_base_completo = Pipeline(steps=[
    ('pre', preprocesador),
    ('clf', model_base)
])

# 4. Definir la grilla de hiperparámetros con el prefijo 'clf__'
param_grid = {
    'clf__n_estimators': [100, 200, 300],
    'clf__learning_rate': [0.05, 0.08, 0.1],
    'clf__max_depth': [3, 4, 5]
}

# 5. Configurar el Grid Search apuntando al nuevo pipeline
grid_search_base = GridSearchCV(
    estimator=pipeline_base_completo,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

# 7. Ejecutar la búsqueda con los datos crudos
grid_search_base.fit(X_train, 1- y_train)

Fitting 5 folds for each of 27 candidates, totalling 135 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...sample=0.8))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'clf__learning_rate': [0.05, 0.08, ...], 'clf__max_depth': [3, 4, ...], 'clf__n_estimators': [100, 200, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is

In [11]:
# --- Resultados para el Modelo Smishing ---
print("=== RESULTADOS MODELO SMISHING ===")
print(f"Mejores hiperparámetros: {grid_search_sm.best_params_}")
print(f"Mejor F1 (Cross-Validation): {grid_search_sm.best_score_:.4f}\n")

# --- Resultados para el Modelo Base ---
print("=== RESULTADOS MODELO BASE ===")
print(f"Mejores hiperparámetros: {grid_search_base.best_params_}")
print(f"Mejor F1 (Cross-Validation): {grid_search_base.best_score_:.4f}")

=== RESULTADOS MODELO SMISHING ===
Mejores hiperparámetros: {'clf__learning_rate': 0.08, 'clf__max_depth': 5, 'clf__n_estimators': 300}
Mejor F1 (Cross-Validation): 0.9815

=== RESULTADOS MODELO BASE ===
Mejores hiperparámetros: {'clf__learning_rate': 0.1, 'clf__max_depth': 5, 'clf__n_estimators': 200}
Mejor F1 (Cross-Validation): 0.9963


## 7. Inferencia sobre transacciones nuevas

### 7a. Posteriors y verosimilitudes

Para recuperar la **verosimilitud** P(x | tipo_k) a partir de la posterior
que devuelve `predict_proba`, corregimos por el prior de entrenamiento:

```
P(x | tipo_k)  ∝  P(tipo_k | x)  /  prior_pos_k
```

Esto elimina el sesgo de prevalencia absorbido durante el entrenamiento.

In [24]:
def score_transaccion_grid(x_row: pd.DataFrame,
                           grid_t0: dict,
                           grid_t1: dict,
                           w_tipo0: float = 0.5,
                           w_tipo1: float = 0.5) -> dict:
    """
    Calcula los scores para una única transacción nueva evaluando la sorpresa
    de Shannon unificada para el monitoreo de regímenes atípicos.

    Parámetros
    ----------
    x_row        : DataFrame de 1 fila con las columnas originales/crudas (sin procesar)
    grid_t0      : dict del modelo Base C1 {'grid': grid_search_base, 'threshold': 0.5, 'prior_pos': p0}
    grid_t1      : dict del modelo Smishing C2 {'grid': grid_search_sm, 'threshold': 0.5, 'prior_pos': p1}
    w_tipo0      : peso/prevalencia del tipo 0 en el mes actual (de Intelligence)
    w_tipo1      : peso/prevalencia del tipo 1 en el mes actual (de Intelligence)
    """
    assert abs(w_tipo0 + w_tipo1 - 1.0) < 1e-6, "w_tipo0 + w_tipo1 debe ser 1"

    # 1. Extraer los mejores pipelines estimados por la grilla
    pipeline_t0 = grid_t0['grid'].best_estimator_
    pipeline_t1 = grid_t1['grid'].best_estimator_

    # 2. Alinear y filtrar las columnas que el ColumnTransformer interno necesita
    cols_necesarias = pipeline_t1.named_steps['pre'].feature_names_in_
    x_input = x_row[cols_necesarias]

    # 3. Calcular probabilidades posteriores de fraude P(Fraude_k | x)
    post0 = pipeline_t0.predict_proba(x_input)[:, 1][0]
    post1 = pipeline_t1.predict_proba(x_input)[:, 1][0]

    # 4. Recuperar los priores históricos de entrenamiento
    prior0 = grid_t0['prior_pos']
    prior1 = grid_t1['prior_pos']

    # 5. Calcular verosimilitudes proporcionales
    like0 = post0 / prior0 if prior0 > 0 else post0
    like1 = post1 / prior1 if prior1 > 0 else post1

    # 6. Likelihood Ratio (Contraste de hipótesis: T1 vs T0)
    LR = like1 / like0 if like0 > 0 else np.inf

    # 7. Cómputo de la densidad estimada del modelo de producción P(e | modelo)
    p_evento_modelo = (w_tipo0 * like0) + (w_tipo1 * like1)

    # 8. Monitoreo de Sorpresa de Shannon (en nats)
    eps = 1e-10  # Previene indeterminaciones matemáticas log(0)
    surprise_score = -np.log(p_evento_modelo + eps)

    # 9. Alertas de negocio por thresholds tradicionales
    alerta0 = post0 >= grid_t0['threshold']
    alerta1 = post1 >= grid_t1['threshold']

    # 10. Alerta por sorpresa (S > 4 nats) -> Desviación drástica del patrón de fraude conocido
    alerta_sorpresa = surprise_score > 4.0

    # 11. Alerta unificada final (Suma lógica de los modelos y el radar de sorpresa)
    alerta_any = alerta0 or alerta1 or alerta_sorpresa

    return {
        'post_tipo0': round(post0, 6),
        'post_tipo1': round(post1, 6),
        'like_tipo0': round(like0, 6),
        'like_tipo1': round(like1, 6),
        'LR(tipo1/tipo0)': round(LR, 4),
        'surprise_score_nats': round(surprise_score, 4),
        'alerta_tipo0': bool(alerta0),
        'alerta_tipo1': bool(alerta1),
        'alerta_sorpresa': bool(alerta_sorpresa),
        'alerta_any': bool(alerta_any)
    }

In [25]:
# Calcular los priores reales (proporción de positivos en el dataset de entrenamiento)
prior_base = (1 - df_merger['cluster_fraude']).mean() # Proporción de legítimos
prior_smishing = df_merger['cluster_fraude'].mean()   # Proporción de fraudes

# Empaquetamos en diccionarios para la función
modelo_t0_dict = {
    'grid': grid_search_base,
    'threshold': 0.5, # Ajusta este valor según tus necesidades de negocio
    'prior_pos': prior_base
}

modelo_t1_dict = {
    'grid': grid_search_sm,
    'threshold': 0.5, # Ajusta este valor según tus necesidades de negocio
    'prior_pos': prior_smishing
}

# Tomamos la primera fila del set de testeo como DataFrame de 1 fila
fila_nueva = X_test.iloc[[0]]

# Ejecutamos la función
res = score_transaccion_grid(fila_nueva, modelo_t0_dict, modelo_t1_dict)
print(res)

{'post_tipo0': np.float64(4e-06), 'post_tipo1': np.float64(0.999999), 'like_tipo0': np.float64(5e-06), 'like_tipo1': np.float64(5.757209), 'LR(tipo1/tipo0)': np.float64(1245323.8465), 'surprise_score_nats': np.float64(-1.0573), 'alerta_tipo0': False, 'alerta_tipo1': True, 'alerta_sorpresa': False, 'alerta_any': True}


In [23]:
fila_nueva

,monto_log,secuencia_24h,terminal_risk_score,canal,dispositivo,hora_bin
5427,9.993862,3,0.250119,online,registrado_reciente,manana


In [15]:
y_test.iloc[[0]]

5427    1.0
Name: cluster_fraude, dtype: float64

### 7b. Scoring en batch sobre un DataFrame completo

In [26]:
import numpy as np
import pandas as pd

def score_batch_grid(df_nuevas: pd.DataFrame,
                     grid_t0: dict,
                     grid_t1: dict,
                     w_tipo0: float = 0.5,
                     w_tipo1: float = 0.5) -> pd.DataFrame:
    """
    Calcula scores masivos para un lote de transacciones evaluando la sorpresa
    de Shannon unificada basándose en la mezcla de verosimilitudes (likes).

    Parámetros
    ----------
    df_nuevas    : DataFrame con los datos crudos/originales de las nuevas operaciones
    grid_t0      : dict del modelo Base C1 {'grid': grid_search_base, 'threshold': 0.5, 'prior_pos': p0}
    grid_t1      : dict del modelo Smishing C2 {'grid': grid_search_sm, 'threshold': 0.5, 'prior_pos': p1}
    w_tipo0      : peso/prevalencia del tipo 0 en el mes actual
    w_tipo1      : peso/prevalencia del tipo 1 en el mes actual
    """
    assert abs(w_tipo0 + w_tipo1 - 1.0) < 1e-6, "Los pesos w deben sumar 1"

    # 1. Extraer los mejores pipelines del GridSearch
    pipeline_t0 = grid_t0['grid'].best_estimator_
    pipeline_t1 = grid_t1['grid'].best_estimator_

    # 2. Alinear y filtrar las columnas que el ColumnTransformer espera
    cols_necesarias = pipeline_t1.named_steps['pre'].feature_names_in_
    df_input = df_nuevas[cols_necesarias]

    # 3. Predicciones masivas (probabilidades posteriores)
    post0 = pipeline_t0.predict_proba(df_input)[:, 1]
    post1 = pipeline_t1.predict_proba(df_input)[:, 1]

    # 4. Recuperar priores de entrenamiento
    prior0 = grid_t0['prior_pos']
    prior1 = grid_t1['prior_pos']

    # 5. Calcular verosimilitudes proporcionales
    like0 = post0 / prior0 if prior0 > 0 else post0
    like1 = post1 / prior1 if prior1 > 0 else post1

    # 6. Likelihood Ratio unificado (Contraste T1 / T0)
    LR = np.divide(
        like1,
        like0,
        out=np.full_like(like1, np.inf),
        where=like0 > 0
    )

    # 7. Cómputo vectorial de la densidad estimada P(e | modelo) usando likes
    p_evento_modelo = (w_tipo0 * like0) + (w_tipo1 * like1)

    # 8. Sorpresa de Shannon unificada (en nats)
    eps = 1e-10
    surprise_score = -np.log(p_evento_modelo + eps)

    # 9. Alertas tradicionales por threshold
    alerta0 = post0 >= grid_t0['threshold']
    alerta1 = post1 >= grid_t1['threshold']

    # 10. Alerta por sorpresa (S > 4 nats) -> Operación legítima atípica
    alerta_sorpresa = surprise_score > 4.0

    # 11. Alerta unificada final (Gatilla si hay sospecha tradicional O si la sorpresa es alta)
    alerta_any = alerta0 | alerta1 | alerta_sorpresa

    # 12. Armar DataFrame de resultados manteniendo el índice original
    return pd.DataFrame({
        'post_tipo0': np.round(post0, 6),
        'post_tipo1': np.round(post1, 6),
        'like_tipo0': np.round(like0, 6),
        'like_tipo1': np.round(like1, 6),
        'LR(tipo1/tipo0)': np.round(LR, 4),
        'surprise_score_nats': np.round(surprise_score, 4),
        'alerta_tipo0': alerta0,
        'alerta_tipo1': alerta1,
        'alerta_sorpresa': alerta_sorpresa,
        'alerta_any': alerta_any
    }, index=df_nuevas.index)

In [27]:
df_scores_test = score_batch_grid(
    df_nuevas=X_test,
    grid_t0=modelo_t0_dict,
    grid_t1=modelo_t1_dict
)

print(f"Alertas totales: {df_scores_test['alerta_any'].sum()} ({df_scores_test['alerta_any'].mean()*100:.2f}%)")
df_scores_test

Alertas totales: 290 (100.00%)


,post_tipo0,post_tipo1,like_tipo0,like_tipo1,LR(tipo1/tipo0),surprise_score_nats,alerta_tipo0,alerta_tipo1,alerta_sorpresa,alerta_any
5427,0.000004,0.999999,0.000005,5.757209,1.245324e+06,-1.0573,False,True,False,True
2612,1.000000,0.000000,1.210207,0.000000,0.000000e+00,0.5024,True,False,False,True
1134,1.000000,0.000000,1.210207,0.000000,0.000000e+00,0.5024,True,False,False,True
4640,0.998355,0.000568,1.208216,0.003268,2.700000e-03,0.5013,True,False,False,True
1137,1.000000,0.000000,1.210207,0.000000,0.000000e+00,0.5024,True,False,False,True
...,...,...,...,...,...,...,...,...,...,...
5554,0.996595,0.000825,1.206086,0.004750,3.900000e-03,0.5018,True,False,False,True
2508,0.995140,0.005043,1.204326,0.029035,2.410000e-02,0.4834,True,False,False,True
3549,1.000000,0.000000,1.210207,0.000000,0.000000e+00,0.5024,True,False,False,True
1358,1.000000,0.000000,1.210207,0.000000,0.000000e+00,0.5024,True,False,False,True


In [33]:
df_scores_test[df_scores_test['surprise_score_nats'] >= 4]

,post_tipo0,post_tipo1,like_tipo0,like_tipo1,LR(tipo1/tipo0),surprise_score_nats,alerta_tipo0,alerta_tipo1,alerta_sorpresa,alerta_any


In [38]:
df_scores_M19 = score_batch_grid(
    df_nuevas=df[(df['etiquetado'] == True) & (df['mes'] == 19)][columnas_esperadas],
    grid_t0=modelo_t0_dict,
    grid_t1=modelo_t1_dict
)

df_scores_M19

,post_tipo0,post_tipo1,like_tipo0,like_tipo1,LR(tipo1/tipo0),surprise_score_nats,alerta_tipo0,alerta_tipo1,alerta_sorpresa,alerta_any
900239,0.002048,0.999110,0.002479,5.752088,2320.2593,-1.0568,False,True,False,True
900240,0.002627,0.998549,0.003179,5.748861,1808.4056,-1.0564,False,True,False,True
900241,0.000018,0.999999,0.000022,5.757208,261604.6957,-1.0573,False,True,False,True
900244,0.896790,0.079879,1.085302,0.459879,0.4237,0.2580,True,False,False,True
900248,0.000012,0.999998,0.000014,5.757204,407399.0734,-1.0573,False,True,False,True
...,...,...,...,...,...,...,...,...,...,...
947582,0.000015,0.999998,0.000018,5.757204,324949.1090,-1.0573,False,True,False,True
947584,0.000096,0.999978,0.000116,5.757090,49622.4421,-1.0573,False,True,False,True
947585,0.000294,0.999904,0.000356,5.756662,16155.7222,-1.0573,False,True,False,True
947586,0.002142,0.998902,0.002592,5.750890,2218.4433,-1.0567,False,True,False,True


In [41]:
df_scores_M19.describe()

,post_tipo0,post_tipo1,like_tipo0,like_tipo1,LR(tipo1/tipo0),surprise_score_nats
count,14319.000000,14319.000000,14319.000000,14319.000000,1.431900e+04,14319.000000
mean,0.027400,0.972860,0.033160,5.600962,9.978590e+05,-1.016078
std,0.158560,0.159325,0.191890,0.917269,4.401808e+06,0.246086
min,0.000000,0.000000,0.000000,0.000000,0.000000e+00,-1.058200
25%,0.000013,0.999783,0.000016,5.755968,9.187574e+03,-1.057300
50%,0.000043,0.999991,0.000052,5.757160,1.101560e+05,-1.057300
75%,0.000518,0.999998,0.000626,5.757202,3.535862e+05,-1.057200
max,1.000000,1.000000,1.210207,5.757214,5.863387e+07,0.502900


In [40]:
df[(df['etiquetado'] == True) & (df['mes'] == 19)]

,op_id,mes,canal,hora_bin,dispositivo,monto_log,terminal_risk_score,secuencia_24h,etiquetado,es_fraude
900239,900240,19,presencial,tarde,conocido,7.944736,0.130966,2,True,False
900240,900241,19,presencial,tarde,conocido,9.115504,0.229485,4,True,False
900241,900242,19,transferencia,noche,conocido,9.490015,0.071279,2,True,False
900244,900245,19,online,tarde,conocido,7.659744,0.506703,2,True,False
900248,900249,19,online,manana,conocido,7.430267,0.184106,1,True,False
...,...,...,...,...,...,...,...,...,...,...
947582,947583,19,online,manana,conocido,8.750480,0.278023,5,True,False
947584,947585,19,atm,tarde,conocido,7.919713,0.282081,4,True,False
947585,947586,19,presencial,manana,conocido,7.232501,0.116362,5,True,False
947586,947587,19,presencial,tarde,conocido,8.578720,0.347487,3,True,False


In [39]:
df_scores_M19[df_scores_M19['surprise_score_nats'] >= 4]

,post_tipo0,post_tipo1,like_tipo0,like_tipo1,LR(tipo1/tipo0),surprise_score_nats,alerta_tipo0,alerta_tipo1,alerta_sorpresa,alerta_any


In [20]:
# 1. Crear el directorio si no existe
os.makedirs('datos/v6/modelos/clasificacion_fraude', exist_ok=True)

# 2. Agrupar tus nuevos diccionarios optimizados con sus respectivas claves
# Usamos nombres claros para identificar el modelo Base (tipo0) y Smishing (tipo1)
nuevos_modelos_guardar = {
    'C1_gbm_base': modelo_t0_dict,
    'C2_gbm_smishing': modelo_t1_dict
}

# 3. Iterar y guardar usando joblib
for key, obj in nuevos_modelos_guardar.items():
    path = f'datos/v6/modelos/clasificacion_fraude/{key}.pkl'

    # Guardamos el diccionario completo (incluye el GridSearch + priores + thresholds)
    joblib.dump(obj, path)
    print(f"Guardado exitosamente: {path}")

print("\nSerialización completa con Joblib.")

Guardado exitosamente: datos/v6/modelos/clasificacion_fraude/C1_gbm_base.pkl
Guardado exitosamente: datos/v6/modelos/clasificacion_fraude/C2_gbm_smishing.pkl

Serialización completa con Joblib.
